In [2]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


data = pd.read_csv('data/application_train_FE_baked.csv')

In [10]:
# New Features

def add_engineered_features(df):
    eps = 1e-6
    df = df.copy()
    df["credit_to_income"]   = df["AMT_CREDIT"] / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_income"]  = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_credit"]  = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + eps)
    df["income_per_member"]  = df["AMT_INCOME_TOTAL"] / np.maximum(df["CNT_FAM_MEMBERS"], 1)
    df["income_per_child"]   = df["AMT_INCOME_TOTAL"] / (1 + df["CNT_CHILDREN"])
    df["dsr_monthly"]        = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"]/12.0 + eps)

    df["age_years"]          = -df["DAYS_BIRTH"] / 365.25
    df["reg_years"]          = -df["DAYS_REGISTRATION"] / 365.25
    df["id_years"]           = -df["DAYS_ID_PUBLISH"] / 365.25
    df["phone_years"]        = -df["DAYS_LAST_PHONE_CHANGE"] / 365.25
    df["contactability"]     = df["FLAG_PHONE"] + df["FLAG_EMAIL"]

    df["prev_interest_per_credit"] = df["prev_interest_mean"] / (df["prev_amt_credit_mean"] + eps)
    df["prev_payments_ratio"]      = df["prev_last3_cnt_payment_mean"] / (df["prev_cnt_payment_mean"] + eps)
    df["recent_approval_momentum"] = df["prev_last3_approved_rate"] - df["prev_last5_approved_rate"]
    df["recent_credit_growth"]     = df["prev_last3_amt_credit_mean"] - df["prev_last5_amt_credit_mean"]
    df["prev_rate_spread"]         = df["prev_rate_mean_all"] - df["prev_rate_med_all"]

    df["region_pop_log"]     = np.log1p(df["REGION_POPULATION_RELATIVE"])
    df["region_rating_x_pop"] = df["REGION_RATING"] * df["REGION_POPULATION_RELATIVE"]

    df["log_income"]         = np.log1p(df["AMT_INCOME_TOTAL"])
    df["log_credit"]         = np.log1p(df["AMT_CREDIT"])
    df["log_annuity"]        = np.log1p(df["AMT_ANNUITY"])
    df["ext2_sq"]            = df["EXT_SOURCE_2"] ** 2
    return df

data_fe = add_engineered_features(data)

c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [14]:
# Make random split, stratified split by target variable, and stratified split by region rating, debt ratio, and target
train_data_random, val_data_random = train_test_split(data_fe, test_size=0.2, random_state=42)
train_data_stratified, val_data_stratified = train_test_split(data_fe, test_size=0.2, random_state=42, stratify=data_fe['TARGET'])

# # For custom stratification, create bins for continuous variables first
# data['REGION_RATING_bins'] = pd.qcut(data['REGION_RATING'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')
# data['debt_ratio_bins'] = pd.qcut(data['debt_ratio'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')

# # Create a combined stratification column
# data['strat_col'] = data['REGION_RATING_bins'].astype(str) + '_' + data['debt_ratio_bins'].astype(str) + '_' + data['TARGET'].astype(str)

# # Now stratify by the combined column
# train_data_custom, val_data_custom = train_test_split(data, test_size=0.2, random_state=42, stratify=data['strat_col'])

In [19]:
# Metrics
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def recall(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    possible_positives = np.sum(y_true == 1)
    return true_positives / possible_positives if possible_positives > 0 else 0

def precision(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    predicted_positives = np.sum(y_pred == 1)
    return true_positives / predicted_positives if predicted_positives > 0 else 0

def specificity(y_true, y_pred):
    true_negatives = np.sum((y_true == 0) & (y_pred == 0))
    possible_negatives = np.sum(y_true == 0)
    return true_negatives / possible_negatives if possible_negatives > 0 else 0

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    return 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

def roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    P = np.sum(y_true == 1)
    N = np.sum(y_true == 0)
    if P == 0 or N == 0:
        return 0.5
    order = np.argsort(-y_score, kind='mergesort')
    y_sorted = y_true[order]
    s_sorted = y_score[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    changes = np.where(np.diff(s_sorted) != 0)[0]
    cut_idx = np.r_[changes, len(s_sorted) - 1]
    tpr = tp[cut_idx] / P
    fpr = fp[cut_idx] / N
    tpr = np.r_[0.0, tpr, 1.0]
    fpr = np.r_[0.0, fpr, 1.0]

    return float(np.trapz(tpr, fpr))

In [16]:
# Cross-validation
def cross_validate(model, X, y, cv=5):
    fold_size = len(X) // cv
    metrics = {'accuracy': [], 'recall': [], 'precision': [], 'specificity': [], 'f1_score': [], 'roc_auc': []}
    
    for fold in range(cv):
        start = fold * fold_size
        end = (fold + 1) * fold_size if fold != cv - 1 else len(X)
        
        X_val_fold = X[start:end]
        y_val_fold = y[start:end]
        X_train_fold = np.concatenate([X[:start], X[end:]], axis=0)
        y_train_fold = np.concatenate([y[:start], y[end:]], axis=0)
        
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        if hasattr(model, "predict_proba"):
            y_pred_prob = model.predict_proba(X_val_fold)[:, 1]
        else:
            y_pred_prob = model.decision_function(X_val_fold)
            y_pred_prob = (y_pred_prob - y_pred_prob.min()) / (y_pred_prob.max() - y_pred_prob.min())
        
        metrics['accuracy'].append(accuracy(y_val_fold, y_pred))
        metrics['recall'].append(recall(y_val_fold, y_pred))
        metrics['precision'].append(precision(y_val_fold, y_pred))
        metrics['specificity'].append(specificity(y_val_fold, y_pred))
        metrics['f1_score'].append(f1_score(y_val_fold, y_pred))
        metrics['roc_auc'].append(roc_auc(y_val_fold, y_pred_prob))
    
    # Calculate average coefficients accross folds if applicable
    coefs = None
    if hasattr(model, 'coef_'):
        coefs = model.coef_
    avg_metrics = {key: np.mean(value) for key, value in metrics.items()}
    return avg_metrics, coefs

In [17]:
# Models

# Logistic Regression
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')

# SVM
svm_model = SVC(probability=True, class_weight='balanced', max_iter=1000)

# LDA
lda_model = LDA()


In [7]:
# Run Models
# Define feature sets for different models
target = "TARGET"

# Basic demographic and financial features
predictors1 = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "debt_ratio",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE",
    "REGION_RATING", "DAYS_BIRTH", "LIVE_CITY_NOT_WORK_CITY"
]

# Previous application statistics
predictors2 = [
    "prev_app_count", "prev_approved_rate", "prev_refused_rate",
    "prev_amt_credit_mean", "prev_cnt_payment_mean", "prev_ann_to_credit_mean",
    "prev_interest_mean", "prev_rate_mean_all", "prev_share_mean_all"
]

# Recent application history features
predictors3 = [
    "prev_last3_n", "prev_last3_approved_rate", "prev_last3_amt_credit_mean",
    "prev_last3_cnt_payment_mean", "prev_last3_ann_to_credit_mean",
    "prev_last5_n", "prev_last5_approved_rate", "prev_last5_cnt_payment_mean"
]

# Document and registration features
predictors4 = [
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "FLAG_PHONE", "FLAG_EMAIL", "LIVE_CITY_NOT_WORK_CITY",
    "REGION_POPULATION_RELATIVE", "REGION_RATING"
]

# Contract and property features
predictors5 = [
    "NAME_CONTRACT_TYPE_Cash.loans", "NAME_CONTRACT_TYPE_Revolving.loans",
    "FLAG_OWN_REALTY_Y", "FLAG_OWN_CAR_Y",
    "NAME_HOUSING_TYPE_House...apartment", "NAME_HOUSING_TYPE_Rented.apartment",
    "NAME_HOUSING_TYPE_With.parents", "AMT_CREDIT", "AMT_ANNUITY"
]

# Income and education features
predictors6 = [
    "NAME_INCOME_TYPE_Working", "NAME_INCOME_TYPE_Commercial.associate",
    "NAME_INCOME_TYPE_Pensioner", "NAME_INCOME_TYPE_State.servant",
    "NAME_EDUCATION_TYPE_Secondary...secondary.special",
    "NAME_EDUCATION_TYPE_Higher.education", "NAME_EDUCATION_TYPE_Lower.secondary",
    "AMT_INCOME_TOTAL", "REGION_RATING"
]

# Combined important features
predictors7 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL",
    "debt_ratio", "prev_approved_rate", "prev_app_count",
    "DAYS_BIRTH", "DAYS_REGISTRATION", "REGION_RATING"
]

# Most important features subset
predictors8 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_INCOME_TOTAL", "debt_ratio", "prev_approved_rate"
]

feature_sets = {
    'Model_1': predictors1,
    'Model_2': predictors2,
    'Model_3': predictors3,
    'Model_4': predictors4,
    'Model_5': predictors5,
    'Model_6': predictors6,
    'Model_7': predictors7,
    'Model_8': predictors8
}

# Example of running cross-validation on Logistic Regression with Model 1 features
X = train_data_random[feature_sets['Model_1']].values
y = train_data_random[target].values
log_reg_metrics1 = cross_validate(log_reg, X, y, cv=5)
log_reg_metrics1


X = train_data_random[feature_sets['Model_2']].values
log_reg_metrics2 = cross_validate(log_reg, X, y, cv=5)
log_reg_metrics2

X = train_data_random[feature_sets['Model_3']].values
log_reg_metrics3 = cross_validate(log_reg, X, y, cv=5)
log_reg_metrics3

X = train_data_random[feature_sets['Model_4']].values
log_reg_metrics4 = cross_validate(log_reg, X, y, cv=5)
log_reg_metrics4

X = train_data_random[feature_sets['Model_5']].values
log_reg_metrics5 = cross_validate(log_reg, X, y, cv=5)
log_reg_metrics5

X = train_data_random[feature_sets['Model_6']].values
log_reg_metrics6 = cross_validate(log_reg, X, y, cv=5)
log_reg_metrics6

X = train_data_random[feature_sets['Model_7']].values
log_reg_metrics7 = cross_validate(log_reg, X, y, cv=5)
log_reg_metrics7

X = train_data_random[feature_sets['Model_8']].values
log_reg_metrics8 = cross_validate(log_reg, X, y, cv=5)
log_reg_metrics8




# svc_metrics1 = cross_validate(svm_model, X, y, cv=5)
# svc_metrics1
# lda_metrics1 = cross_validate(lda_model, X, y, cv=5)
# lda_metrics1

({'accuracy': 0.6593364590988665,
  'recall': 0.5882557425959792,
  'precision': 0.1341921685186394,
  'specificity': 0.6655891565348262,
  'f1_score': 0.2185254761468673,
  'roc_auc': 0.6755322384273705},
 array([[-0.52046265, -0.03718621,  0.00172925,  0.32216728, -0.19731096]]))

In [8]:
print(log_reg_metrics1)
print(log_reg_metrics2)
print(log_reg_metrics3)
print(log_reg_metrics4)
print(log_reg_metrics5)
print(log_reg_metrics6)
print(log_reg_metrics7)
print(log_reg_metrics8)

({'accuracy': 0.5974180045447591, 'recall': 0.5895279587412384, 'precision': 0.11442634849427813, 'specificity': 0.5981174220095695, 'f1_score': 0.19164705612091443, 'roc_auc': 0.6264746249388997}, array([[ 0.00060852, -0.15112712,  0.10995411,  0.30566738,  0.07675181,
        -0.11142969, -0.03078039,  0.19910989,  0.26594   ,  0.07808056]]))
({'accuracy': 0.6433180832516283, 'recall': 0.5214217648330761, 'precision': 0.11721534311431799, 'specificity': 0.6540488118089486, 'f1_score': 0.19139902711728782, 'roc_auc': 0.6168165866814477}, array([[-0.09908665, -0.06231818,  0.22492471, -0.24766376,  0.05339873,
        -0.1270122 ,  0.08139867,  0.20184356,  0.06484542]]))
({'accuracy': 0.5530822416666685, 'recall': 0.541865890562221, 'precision': 0.09670475133655523, 'specificity': 0.5540604350474668, 'f1_score': 0.16411384015694702, 'roc_auc': 0.5744767166085316}, array([[ 0.0032623 , -0.00110756, -0.13233805,  0.0322312 , -0.06957491,
        -0.16296084, -0.25681383,  0.02562887]]))

In [18]:
x7_plus = [
    # base “strong compact” (set 7)
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "prev_approved_rate","prev_app_count","DAYS_BIRTH","DAYS_REGISTRATION","REGION_RATING",
    # + engineered
    "credit_to_income","annuity_to_income","annuity_to_credit",
    "income_per_member","dsr_monthly",
    "age_years","reg_years",
    "prev_interest_per_credit","prev_payments_ratio","recent_approval_momentum",
    "prev_rate_spread","region_rating_x_pop","ext2_sq"
]

x8_plus = [
    # base “tiny5” (set 8)
    "EXT_SOURCE_2","AMT_CREDIT","AMT_INCOME_TOTAL","debt_ratio","prev_approved_rate",
    # + engineered
    "credit_to_income","annuity_to_income","annuity_to_credit",
    "income_per_member","dsr_monthly",
    "age_years","reg_years",
    "prev_interest_per_credit","recent_approval_momentum","ext2_sq"
]

X = train_data_random[x7_plus].values
y = train_data_random[target].values
log_reg_metrics9 = cross_validate(log_reg, X, y, cv=5)
print(log_reg_metrics9)

X = train_data_random[x8_plus].values
y = train_data_random[target].values
log_reg_metrics10 = cross_validate(log_reg, X, y, cv=5)
print(log_reg_metrics10)


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

({'accuracy': 0.6541260187020845, 'recall': 0.613152422704111, 'precision': 0.13630286942884232, 'specificity': 0.6577325488172815, 'f1_score': 0.22302198464286754, 'roc_auc': 0.6887531628253221}, array([[-5.06164276e-01, -9.72378546e-02,  1.14810275e-01,
        -5.25678438e-03,  2.97422392e-01, -2.24392139e-01,
        -4.26425115e-02,  2.25874584e-01,  2.79004427e-02,
         6.94315279e-02, -5.05359219e-04, -2.52091810e-02,
        -2.40868012e-05,  5.22341733e-03,  2.06832423e-03,
        -6.18410908e-04, -7.63872490e-05, -3.60035371e-05,
         6.05515263e-05,  3.26703571e-02, -3.46430935e-03,
         1.73451648e-02, -2.06452776e-02]]))


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

({'accuracy': 0.6496081025660126, 'recall': 0.5979663926260403, 'precision': 0.13218548676073189, 'specificity': 0.6541461348743196, 'f1_score': 0.2165060142444765, 'roc_auc': 0.6755986048564374}, array([[-5.68462660e-01, -3.25377988e-02, -1.54371338e-01,
         3.14199612e-01, -1.92421344e-01, -5.25097571e-04,
         2.90005963e-03, -8.93535997e-05,  1.57778618e-01,
        -2.20932852e-04, -4.36474197e-01, -1.94097484e-01,
        -3.80426478e-05,  5.79799059e-02, -4.21429820e-02]]))


In [12]:
# get all columns except target
target = "TARGET"
X_all = train_data_random.drop(columns=[target, 'SK_ID_CURR']).values
y = train_data_random[target].values
log_reg_metrics_all = cross_validate(log_reg, X_all, y, cv=5)
print(log_reg_metrics_all)
svc_metrics_all = cross_validate(svm_model, X_all, y, cv=5)
print(svc_metrics_all)
lda_metrics_all = cross_validate(lda_model, X_all, y, cv=5)
print(lda_metrics_all)


KeyboardInterrupt: 